# w2-analysis — Week 2: Dictionary Regressions & Validation

**Project:** Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews

**Superseded for publication:** This development notebook is retained for provenance. Canonical corrected computations are implemented in `analysis/` and presented in `corrected_results.ipynb`.

Four tasks in one notebook:
- **Task A:** Dictionary-score OLS regressions (full, without explicit eval, section-level)
- **Task B:** Known-groups validation (Islay/peated, sherry, bourbon, high-score, low-score)
- **Task C:** Generic sentiment comparison (VADER)
- **Task D:** Close-reading candidate selection

## 1. Setup & Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from scipy import stats
import json, re, os, warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/mac/Desktop/CSSS594/FinalProject')
os.makedirs('figures', exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
                      'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})
PALETTE = sns.color_palette('tab10')

def save_fig(name):
    for ext in ['pdf', 'png']:
        plt.savefig(f'figures/{name}.{ext}', bbox_inches='tight')
    plt.close()

# --------------------------------------------------------------------
# Load and merge
# --------------------------------------------------------------------
df = pd.read_parquet('data/whiskyfun_tokenized.parquet')
feat = pd.read_parquet('data/v2/whiskyfun_dict_features.parquet')
with open('data/whiskyfun_dictionary_v2.json') as f:
    dictionary = json.load(f)
print(f"Loaded: {len(df):,} reviews, {feat.shape[1]} feature columns, {dictionary['total_terms']} dict terms")

# Drop duplicate metadata columns from feat before merge
meta_overlap = ['score', 'review_year', 'distillery', 'review_length',
                'identity_status', 'match_source']
feat_to_merge = feat.drop(columns=[c for c in meta_overlap if c in feat.columns])
df = df.merge(feat_to_merge, on='dedupe_hash', how='left')
print(f"Merged: {len(df)} rows x {len(df.columns)} columns")

# Category info
CAT_KEYS = list(dictionary['categories'].keys())
# V2 category key -> short_name (matches feature column prefix)
CAT_SHORT = {meta['short_name']: meta['short_name'] for meta in dictionary['categories'].values()}
# Overwrite with explicit mapping from category key to short_name
CAT_SHORT = {key: dictionary['categories'][key]['short_name'] for key in CAT_KEYS}
SHORT_CATS = [CAT_SHORT[c] for c in CAT_KEYS]
FULL_NAMES = [dictionary['categories'][c]['display_label'] for c in CAT_KEYS]
SCOPES = ['review_text', 'nose', 'mouth', 'finish', 'comments', 'nmf']
SCOPE_LABELS = ['Full Text', 'Nose', 'Mouth', 'Finish', 'Comments', 'NMF']

Loaded: 11,149 reviews, 215 feature columns, 247 dict terms
Merged: 11149 rows x 225 columns


## Task A: Dictionary-Score Regressions

### A1. Full Dictionary OLS Regression

In [3]:
# Primary model: all 11 V2 categories on review_text scope, with review_length + year FE
cat_terms_full = ' + '.join([f'{s}_review_text_per1k' for s in SHORT_CATS])
formula_full = f'score ~ {cat_terms_full} + review_length + C(review_year)'

m_full = smf.ols(formula_full, data=df).fit()
print(m_full.summary().tables[1])
print(f"Adj R²: {m_full.rsquared_adj:.4f}, R²: {m_full.rsquared:.4f}, N: {int(m_full.nobs)}")

                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      77.3312      0.290    266.329      0.000      76.762      77.900
C(review_year)[T.2013]          0.0971      0.247      0.394      0.694      -0.386       0.580
C(review_year)[T.2014]          0.4410      0.244      1.804      0.071      -0.038       0.920
C(review_year)[T.2015]         -0.4123      0.251     -1.645      0.100      -0.904       0.079
C(review_year)[T.2016]          0.3499      0.257      1.359      0.174      -0.155       0.855
C(review_year)[T.2017]          0.1339      0.239      0.561      0.575      -0.334       0.602
C(review_year)[T.2018]          0.0598      0.235      0.254      0.799      -0.402       0.521
C(review_year)[T.2019]          0.3881      0.244      1.591      0.112      -0.090       0.866
C(review_year)[T.2020]          0.5537  

### A2. Without Explicit Evaluation

In [4]:
# Drop eval category — tests whether sensory/structural vocabulary retains signal (10 categories)
cats_no_eval = [s for s in SHORT_CATS if s != 'eval']
cat_terms_no_eval = ' + '.join([f'{s}_review_text_per1k' for s in cats_no_eval])
formula_no_eval = f'score ~ {cat_terms_no_eval} + review_length + C(review_year)'

m_no_eval = smf.ols(formula_no_eval, data=df).fit()
print(m_no_eval.summary().tables[1])
print(); print(f"Adj R²: {m_no_eval.rsquared_adj:.4f}, R²: {m_no_eval.rsquared:.4f}")
delta_r2 = m_full.rsquared_adj - m_no_eval.rsquared_adj
print(f"ΔAdj-R² (full - no_eval): {delta_r2:+.4f}  ← eval category contribution")

# Save M3 adj-R² for w4 to read
pd.DataFrame([{
    'model': 'M2_full', 'adj_r2': round(m_full.rsquared_adj, 4),
}, {
    'model': 'M3_no_eval', 'adj_r2': round(m_no_eval.rsquared_adj, 4),
}]).to_csv('data/v2/w2_model_summary.csv', index=False)

                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      78.9403      0.283    278.738      0.000      78.385      79.495
C(review_year)[T.2013]          0.0447      0.251      0.178      0.859      -0.447       0.536
C(review_year)[T.2014]          0.3880      0.249      1.561      0.119      -0.099       0.875
C(review_year)[T.2015]         -0.4549      0.255     -1.784      0.074      -0.955       0.045
C(review_year)[T.2016]          0.3539      0.262      1.351      0.177      -0.159       0.867
C(review_year)[T.2017]          0.0696      0.243      0.287      0.774      -0.406       0.545
C(review_year)[T.2018]          0.1070      0.239      0.447      0.655      -0.362       0.576
C(review_year)[T.2019]          0.3867      0.248      1.559      0.119      -0.100       0.873
C(review_year)[T.2020]          0.4876  

### A3. Section-Level Comparisons

In [5]:
# Run full-dictionary model on each scope, using wordcount_{scope} as length control
# Drop rows where wordcount_{scope} == 0
section_results = []
scope_formulas = {}

for scope, label in zip(SCOPES, SCOPE_LABELS):
    wc_col = f'wordcount_{scope}'
    # Filter to rows with text in this scope
    scope_df = df[df[wc_col] > 0].copy()

    cat_terms_scope = ' + '.join([f'{s}_{scope}_per1k' for s in SHORT_CATS])
    formula = f'score ~ {cat_terms_scope} + {wc_col} + C(review_year)'
    scope_formulas[scope] = formula

    try:
        m = smf.ols(formula, data=scope_df).fit()
        section_results.append({
            'Scope': label,
            'N': int(m.nobs),
            'Adj_R2': round(m.rsquared_adj, 4),
            'R2': round(m.rsquared, 4)
        })
        print(f"{label:<12s}: Adj-R²={m.rsquared_adj:.4f}, N={int(m.nobs)}")
    except Exception as e:
        print(f"{label}: FAILED — {e}")

# Baseline: length + year FE only (review_text scope)
formula_baseline = 'score ~ review_length + C(review_year)'
m_baseline = smf.ols(formula_baseline, data=df).fit()
section_results.append({
    'Scope': 'Baseline (length+year)', 'N': int(m_baseline.nobs),
    'Adj_R2': round(m_baseline.rsquared_adj, 4), 'R2': round(m_baseline.rsquared, 4)
})
print(); print(f"Baseline (length+year): Adj-R²={m_baseline.rsquared_adj:.4f}")

sr_df = pd.DataFrame(section_results)
print(); print("=== R² Comparison Across Scopes ===")
print(sr_df.to_string(index=False))

# Figure: R² comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(sr_df)), sr_df['Adj_R2'], color=[PALETTE[i % 10] for i in range(len(sr_df))], edgecolor='white')
ax.set_xticks(range(len(sr_df))); ax.set_xticklabels(sr_df['Scope'], rotation=30, ha='right')
ax.set_ylabel('Adjusted R²'); ax.set_title('Figure W2-1: R² by Text Scope (Full Dictionary Model)')
for bar, val in zip(bars, sr_df['Adj_R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout(); save_fig('fig_w2_a3_r2_by_scope')

Full Text   : Adj-R²=0.2891, N=11149
Nose        : Adj-R²=0.2075, N=11133
Mouth       : Adj-R²=0.2339, N=11130
Finish      : Adj-R²=0.1300, N=11120
Comments    : Adj-R²=0.0220, N=11100
NMF         : Adj-R²=0.3359, N=11133

Baseline (length+year): Adj-R²=0.1031

=== R² Comparison Across Scopes ===
                 Scope     N  Adj_R2     R2
             Full Text 11149  0.2891 0.2907
                  Nose 11133  0.2075 0.2093
                 Mouth 11130  0.2339 0.2356
                Finish 11120  0.1300 0.1319
              Comments 11100  0.0220 0.0242
                   NMF 11133  0.3359 0.3374
Baseline (length+year) 11149  0.1031 0.1043


### A4. Regression Coefficient Table

In [6]:
# Build clean coefficient table from the full model
coef_table = pd.DataFrame({
    'Coefficient': m_full.params,
    'SE': m_full.bse,
    't': m_full.tvalues,
    'p': m_full.pvalues
})

# Add significance stars
def stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''

coef_table['sig'] = [stars(p) for p in coef_table['p']]
coef_table['Coef (SE)'] = [
    f"{c:.3f}{s} ({se:.3f})" for c, se, s in zip(coef_table['Coefficient'], coef_table['SE'], coef_table['sig'])
]

# Keep only category features + review_length
cat_rows = [r for r in coef_table.index if 'per1k' in r or 'review_length' in r]
print("=== Full Dictionary Model: Category Coefficients ===")
print(coef_table.loc[cat_rows, ['Coef (SE)', 't', 'p']].to_string())

# Save
coef_table.to_csv('data/v2/w2_regression_results.csv')
print(); print("Saved: data/v2/w2_regression_results.csv")
sr_df.to_csv('data/v2/w2_r2_by_scope.csv', index=False)

=== Full Dictionary Model: Category Coefficients ===
                                     Coef (SE)          t              p
fruit_review_text_per1k       0.039*** (0.003)  12.141925   1.034377e-33
floral_review_text_per1k       0.033** (0.010)   3.208271   1.339138e-03
spice_review_text_per1k        -0.018* (0.008)  -2.218128   2.656623e-02
peat_review_text_per1k        0.064*** (0.003)  24.260672  9.676214e-127
sherry_review_text_per1k      0.044*** (0.003)  14.948664   4.857880e-50
oak_review_text_per1k        -0.044*** (0.005)  -9.254026   2.556912e-20
texture_review_text_per1k     0.062*** (0.009)   7.112020   1.213770e-12
mineral_review_text_per1k       0.011* (0.004)   2.433649   1.496312e-02
flaw_review_text_per1k       -0.331*** (0.010) -33.625586  3.671674e-236
structure_review_text_per1k   0.036*** (0.009)   4.153919   3.292710e-05
eval_review_text_per1k        0.094*** (0.005)  19.564634   7.876884e-84
review_length                 0.035*** (0.001)  37.457225  2.661101e-28

## Task B: Known-Groups Validation

### B1. Islay / Peated Groups

In [7]:
# Method A: Distillery-based
ISLAY_DISTS = ['Ardbeg', 'Bowmore', 'Bruichladdich', 'Bunnahabhain',
               'Caol Ila', 'Kilchoman', 'Laphroaig', 'Port Ellen']
df['islay_dist'] = df['distillery'].isin(ISLAY_DISTS)

# Method B: Feature-based (preferred — captures peated style regardless of distillery)
peat_75 = df['peat_review_text_per1k'].quantile(0.75)
df['is_peated'] = df['peat_review_text_per1k'] > peat_75

print(f"Islay by distillery: {df['islay_dist'].sum():,} reviews")
print(f"Peated (peat > P75={peat_75:.1f}/1k): {df['is_peated'].sum():,} reviews")
print(f"Overlap: {((df['islay_dist']) & (df['is_peated'])).sum():,} reviews are Islay AND peated")

# Use feature-based for comparisons
def compare_groups(mask, label, cats_to_show=None):
    """Compare dictionary rates between group (mask=True) and rest."""
    if cats_to_show is None:
        cats_to_show = SHORT_CATS

    results = []
    for short in cats_to_show:
        col = f'{short}_review_text_per1k'
        g1 = df.loc[mask, col].dropna()
        g2 = df.loc[~mask, col].dropna()

        # t-test (Welch's) and Mann-Whitney
        t_stat, t_p = stats.ttest_ind(g1, g2, equal_var=False)
        mw_u, mw_p = stats.mannwhitneyu(g1, g2, alternative='two-sided')

        # Cohen's d
        pooled_std = np.sqrt((g1.std()**2 + g2.std()**2) / 2)
        d = (g1.mean() - g2.mean()) / pooled_std if pooled_std > 0 else 0

        results.append({
            'Category': short,
            f'{label}_mean': round(g1.mean(), 2),
            'Other_mean': round(g2.mean(), 2),
            'Diff': round(g1.mean() - g2.mean(), 2),
            "Cohen_d": round(d, 3),
            't_p': round(t_p, 4),
            'sig': stars(t_p)
        })
    return pd.DataFrame(results)

print(); print("=== Peated vs Non-Peated ===")
peat_results = compare_groups(df['is_peated'], 'Peated')
print(peat_results.to_string(index=False))
peat_results.to_csv('data/v2/w2_known_groups_peat.csv', index=False)

Islay by distillery: 2,215 reviews
Peated (peat > P75=17.3/1k): 2,786 reviews
Overlap: 1,382 reviews are Islay AND peated

=== Peated vs Non-Peated ===
 Category  Peated_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit        15.05       19.42 -4.37   -0.318 0.0000 ***
   floral         1.15        1.96 -0.81   -0.198 0.0000 ***
    spice         3.44        3.41  0.03    0.006 0.7767    
     peat        35.93        3.47 32.47    2.584 0.0000 ***
   sherry         8.08       13.31 -5.23   -0.367 0.0000 ***
      oak         4.57        7.08 -2.51   -0.293 0.0000 ***
  texture         3.09        2.79  0.30    0.060 0.0069  **
  mineral         8.42        8.05  0.37    0.038 0.0777    
     flaw         1.74        1.75 -0.01   -0.002 0.9116    
structure         3.42        2.92  0.50    0.099 0.0000 ***
     eval        10.26       10.37 -0.11   -0.012 0.5939    


### B2-B3. Sherry & Bourbon Cask Groups

In [8]:
# Sherry: top quartile of sherry per1k
sherry_75 = df['sherry_review_text_per1k'].quantile(0.75)
df['is_sherried'] = df['sherry_review_text_per1k'] > sherry_75
print(f"Sherried (sherry > P75={sherry_75:.1f}/1k): {df['is_sherried'].sum():,} reviews")

# Bourbon: high oak, low sherry
oak_75 = df['oak_review_text_per1k'].quantile(0.75)
sherry_median = df['sherry_review_text_per1k'].median()
df['is_bourbon'] = (df['oak_review_text_per1k'] > oak_75) & (df['sherry_review_text_per1k'] < sherry_median)
print(f"Bourbon cask (oak>P75={oak_75:.1f}, sherry<median={sherry_median:.1f}): {df['is_bourbon'].sum():,} reviews")

print(); print("=== Sherried vs Non-Sherried ===")
sherry_results = compare_groups(df['is_sherried'], 'Sherried')
print(sherry_results.to_string(index=False))
sherry_results.to_csv('data/v2/w2_known_groups_sherry.csv', index=False)

print(); print("=== Bourbon Cask vs Rest ===")
bourbon_results = compare_groups(df['is_bourbon'], 'Bourbon')
print(bourbon_results.to_string(index=False))
bourbon_results.to_csv('data/v2/w2_known_groups_bourbon.csv', index=False)

Sherried (sherry > P75=17.8/1k): 2,783 reviews
Bourbon cask (oak>P75=9.4, sherry<median=6.7): 1,427 reviews

=== Sherried vs Non-Sherried ===
 Category  Sherried_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit          14.45       19.62 -5.17   -0.374 0.0000 ***
   floral           1.36        1.89 -0.54   -0.131 0.0000 ***
    spice           3.89        3.26  0.63    0.113 0.0000 ***
     peat           7.63       12.89 -5.26   -0.342 0.0000 ***
   sherry          34.08        4.66 29.42    2.660 0.0000 ***
      oak           5.70        6.70 -1.00   -0.113 0.0000 ***
  texture           2.16        3.10 -0.94   -0.195 0.0000 ***
  mineral           6.48        8.70 -2.22   -0.235 0.0000 ***
     flaw           1.66        1.78 -0.12   -0.029 0.1794    
structure           3.25        2.98  0.27    0.056 0.0102   *
     eval          10.04       10.44 -0.41   -0.045 0.0375   *

=== Bourbon Cask vs Rest ===
 Category  Bourbon_mean  Other_mean   Diff  Cohen_d    t_p sig
    fruit

These are text-defined groups, not verified cask types:


Sherried: `sherry_review_text_per1k` > 75th percentile (17.8/1k); N = 2,783
Bourbon cask: `oak` > P75 (9.4/1k) and `sherry` < median (6.7/1k); N = 1,427

“Sherried” ≈ reviews that talk a lot about sherry; “bourbon cask” ≈ reviews with high oak + relatively low sherry language. That aligns with how Serge (Whiskyfun) writes about maturation, but it is still a proxy from vocabulary, not a ground-truth cask register.

However, there are many bottles whose cask types were not clearly specified (such as "wood", "hogshead", "refill cask")

A mean of `34.08` for sherry in the `sherried group`: on average, those reviews use about 34 sherry-dictionary terms per 1,000 words of review text

A mean of `1.48` for sherry in the `bourbon group`: on average, those reviews use about 1.48 sherry-dictionary terms per 1,000 words of review text

### B4-B5. High-Score & Low-Score Groups

In [9]:
df['high_score'] = df['score'] >= 90
df['low_score'] = df['score'] <= 75
print(f"High-score (>=90): {df['high_score'].sum():,} reviews")
print(f"Low-score (<=75): {df['low_score'].sum():,} reviews")

print(); print("=== High-Score (>=90) vs Rest ===")
high_results = compare_groups(df['high_score'], 'HighScore')
print(high_results.to_string(index=False))
high_results.to_csv('data/v2/w2_known_groups_highscore.csv', index=False)

print(); print("=== Low-Score (<=75) vs Rest ===")
low_results = compare_groups(df['low_score'], 'LowScore')
print(low_results.to_string(index=False))
low_results.to_csv('data/v2/w2_known_groups_lowscore.csv', index=False)

High-score (>=90): 2,951 reviews
Low-score (<=75): 359 reviews

=== High-Score (>=90) vs Rest ===
 Category  HighScore_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit           16.34       19.04 -2.70   -0.192 0.0000 ***
   floral            1.92        1.70  0.21    0.051 0.0168   *
    spice            2.76        3.66 -0.90   -0.175 0.0000 ***
     peat           16.87        9.68  7.19    0.413 0.0000 ***
   sherry           12.20       11.93  0.27    0.017 0.4204    
      oak            4.71        7.08 -2.37   -0.280 0.0000 ***
  texture            3.52        2.63  0.89    0.170 0.0000 ***
  mineral            7.80        8.27 -0.47   -0.048 0.0228   *
     flaw            0.87        2.07 -1.20   -0.304 0.0000 ***
structure            3.72        2.80  0.92    0.183 0.0000 ***
     eval           11.36        9.97  1.39    0.150 0.0000 ***

=== Low-Score (<=75) vs Rest ===
 Category  LowScore_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit          11.92       18.54 

### B6. Known-Groups Summary Figure

In [10]:
# Combined figure: key category differences across all groups
groups_data = {
    'Peated': ('is_peated', ['peat', 'sherry', 'oak', 'flaw']),
    'Sherried': ('is_sherried', ['sherry', 'oak', 'fruit', 'flaw']),
    'Bourbon': ('is_bourbon', ['oak', 'sherry', 'vanilla_not_in_features_use_oak', 'flaw']),
    'High Score': ('high_score', ['structure', 'fruit', 'eval', 'flaw']),
    'Low Score': ('low_score', ['flaw', 'structure', 'fruit', 'eval']),
}

# Simplified version for figure — 4 groups × key categories
groups_for_fig = {
    'Peated': ('is_peated', ['peat', 'flaw', 'structure', 'eval']),
    'Sherried': ('is_sherried', ['sherry', 'peat', 'structure', 'eval']),
    'High Score': ('high_score', ['structure', 'eval', 'flaw', 'peat']),
    'Low Score': ('low_score', ['flaw', 'eval', 'structure', 'peat']),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (gname, (mask_col, cats)) in zip(axes.flat, groups_for_fig.items()):
    x = np.arange(len(cats)); width = 0.35
    mask = df[mask_col]
    g_means = [df.loc[mask, f'{c}_review_text_per1k'].mean() for c in cats]
    o_means = [df.loc[~mask, f'{c}_review_text_per1k'].mean() for c in cats]

    ax.bar(x - width/2, g_means, width, label=gname, color=PALETTE[1], edgecolor='white')
    ax.bar(x + width/2, o_means, width, label='Other', color=PALETTE[5], edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=10)
    ax.set_ylabel('Mean per 1k tokens'); ax.legend(fontsize=9)
    ax.set_title(f'{gname} (n={mask.sum():,})')

plt.suptitle('Figure W2-2: Known-Groups Validation — Dictionary Feature Comparison', fontsize=14)
plt.tight_layout(); save_fig('fig_w2_b_known_groups')

# Combined results table
all_group_results = pd.concat([
    peat_results.assign(Group='Peated'),
    sherry_results.assign(Group='Sherried'),
    bourbon_results.assign(Group='Bourbon'),
    high_results.assign(Group='HighScore'),
    low_results.assign(Group='LowScore')
], ignore_index=True)
all_group_results.to_csv('data/v2/w2_known_groups.csv', index=False)
print("Combined results saved: data/v2/w2_known_groups.csv")

Combined results saved: data/v2/w2_known_groups.csv


## Task C: Generic Sentiment Comparison

VADER (Valence Aware Dictionary for sEntiment Reasoning) is a lexicon- and rule-based sentiment model designed for short texts, especially social media language. It scores text along positive, negative, neutral, and compound dimensions using a hand-built lexicon plus simple rules (e.g., capitalization, degree words like “very,” punctuation such as “!!!,” and contrastive “but”) so that valence is context-sensitive rather than a naive sum of word polarities.

The lexicon and its valence weights were built from human sentiment ratings, not from a single fixed “training corpus” in the deep-learning sense. Hutto and Gilbert (2014) report that thousands of lexical features (including slang, emoticons, and common social-web expressions) were rated by multiple independent annotators on Amazon Mechanical Turk on a scale from “most negative” to “most positive,” and those ratings were aggregated into the gold-standard valence scores that drive VADER.

C.J. Hutto and Eric Gilbert. “VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text.” Proceedings of the International AAAI Conference on Web and Social Media (ICWSM), 2014.

Here, VADER is applied via NLTK’s `SentimentIntensityAnalyzer` to the original review text (review_text_original in analysis/common.py), i.e., the same substantive content our dictionary uses

### C1. Compute VADER Sentiment

In [11]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Ensure VADER lexicon is available
import nltk
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

analyzer = SentimentIntensityAnalyzer()

# Apply to untokenized review text
def vader_scores(text):
    if pd.isna(text) or str(text).strip() == '':
        return pd.Series({'vader_compound': np.nan, 'vader_pos': np.nan,
                          'vader_neg': np.nan, 'vader_neu': np.nan})
    scores = analyzer.polarity_scores(str(text))
    return pd.Series({'vader_compound': scores['compound'],
                      'vader_pos': scores['pos'],
                      'vader_neg': scores['neg'],
                      'vader_neu': scores['neu']})

print("Computing VADER sentiment on review_text_original...")
vader_df = df['review_text_original'].apply(vader_scores)

for col in ['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']:
    df[col] = vader_df[col]

print(f"VADER compound: mean={df['vader_compound'].mean():.3f}, std={df['vader_compound'].std():.3f}")
print(f"  Positive: mean={df['vader_pos'].mean():.3f}")
print(f"  Negative: mean={df['vader_neg'].mean():.3f}")
print(f"  Neutral:  mean={df['vader_neu'].mean():.3f}")

# Correlation with score
for col in ['vader_compound', 'vader_pos', 'vader_neg']:
    r = df[col].corr(df['score'])
    print(f"  score ~ {col}: r={r:+.4f}")

Computing VADER sentiment on review_text_original...
VADER compound: mean=0.855, std=0.321
  Positive: mean=0.152
  Negative: mean=0.036
  Neutral:  mean=0.812
  score ~ vader_compound: r=+0.2822
  score ~ vader_pos: r=+0.1268
  score ~ vader_neg: r=-0.2470


### C2. Sentiment Regression & R² Comparison

In [16]:
# M0: Baseline (length + year FE)
m0 = smf.ols('score ~ review_length + C(review_year)', data=df).fit()

# M1: VADER sentiment
m1 = smf.ols('score ~ vader_compound + review_length + C(review_year)', data=df).fit()

# Collect R² values
r2_comparison = pd.DataFrame([
    {'Model': 'M0: Baseline (length + year)', 'Adj_R2': round(m0.rsquared_adj, 4), 'R2': round(m0.rsquared, 4), 'N': int(m0.nobs)},
    {'Model': 'M1: VADER Sentiment', 'Adj_R2': round(m1.rsquared_adj, 4), 'R2': round(m1.rsquared, 4), 'N': int(m1.nobs)},
    {'Model': 'M2: Full Dictionary (11 cats)', 'Adj_R2': round(m_full.rsquared_adj, 4), 'R2': round(m_full.rsquared, 4), 'N': int(m_full.nobs)},
    {'Model': 'M3: Dictionary minus Eval (10 cats)', 'Adj_R2': round(m_no_eval.rsquared_adj, 4), 'R2': round(m_no_eval.rsquared, 4), 'N': int(m_no_eval.nobs)},
])

print("=== R² Comparison: Sentiment vs. Dictionary ===")
print(r2_comparison.to_string(index=False))

# Figure
fig, ax = plt.subplots(figsize=(9, 5))
models_short = ['Baseline', 'VADER', 'Full Dict (11)', 'Dict-Eval (10)']
colors = [PALETTE[5], PALETTE[3], PALETTE[0], PALETTE[1]]
bars = ax.bar(models_short, r2_comparison['Adj_R2'], color=colors, edgecolor='white')
ax.set_ylabel('Adjusted R²'); ax.set_title('Figure W2-3: R² Comparison — Generic Sentiment vs. Domain Dictionary')
for bar, val in zip(bars, r2_comparison['Adj_R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}', ha='center', fontsize=11)
plt.tight_layout(); save_fig('fig_w2_c_r2_comparison')

# Save updated features with VADER columns to a new parquet
vader_cols = ['dedupe_hash', 'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']
feat_updated = feat.merge(df[vader_cols], on='dedupe_hash', how='left')
feat_updated.to_parquet('data/v2/whiskyfun_dict_features.parquet', index=False)
print(); print("Updated features saved: data/v2/whiskyfun_dict_features.parquet (now includes VADER columns)")

=== R² Comparison: Sentiment vs. Dictionary ===
                              Model  Adj_R2     R2     N
       M0: Baseline (length + year)  0.1031 0.1043 11149
                M1: VADER Sentiment  0.1609 0.1620 11149
      M2: Full Dictionary (11 cats)  0.2898 0.2913 11149
M3: Dictionary minus Eval (10 cats)  0.2654 0.2670 11149

Updated features saved: data/v2/whiskyfun_dict_features.parquet (now includes VADER columns)


## Task D: Close-Reading Candidate Selection

### D1. Define Selection Functions

In [17]:
def find_reviews(mask, criterion_name, n=3, sort_by=None):
    """Select up to n reviews matching mask, optionally sorted by a column."""
    candidates = df[mask].copy()
    if len(candidates) == 0:
        print(f"  {criterion_name}: NO MATCHES")
        return pd.DataFrame()
    if sort_by:
        candidates = candidates.sort_values(sort_by, ascending=False)
    selected = candidates.head(n).copy()
    selected['selection_criterion'] = criterion_name
    return selected[
        ['dedupe_hash', 'whisky_name_raw', 'distillery', 'score', 'review_text_original',
         'review_year'] +
        [f'{c}_review_text_per1k' for c in SHORT_CATS] +
        ['selection_criterion']
    ]

# Define thresholds
flaw_p90 = df['flaw_review_text_per1k'].quantile(0.90)
fruit_p75 = df['fruit_review_text_per1k'].quantile(0.75)
peat_p75 = df['peat_review_text_per1k'].quantile(0.75)
structure_p75 = df['structure_review_text_per1k'].quantile(0.75)
eval_median = df['eval_review_text_per1k'].median()

print(f"Thresholds: flaw P90={flaw_p90:.1f}, fruit P75={fruit_p75:.1f}, peat P75={peat_p75:.1f}, structure P75={structure_p75:.1f}")

# Term-level regex searches on tokenized review_text
def term_search(terms):
    """Build regex for any of the given terms (word-boundary, case-insensitive)."""
    return df['review_text'].str.contains(
        r'\b(?:' + '|'.join(re.escape(t) for t in terms) + r')\b',
        case=False, regex=True, na=False
    )

has_sulphur = term_search(['sulphur', 'sulfurous', 'sulphuric'])
has_farmyard = term_search(['farmyard', 'farmy', 'barnyard'])
has_overoaked = term_search(['over_oaked', 'planky', 'too_woody', 'over_wooded'])
has_nostalgia = term_search(['old_school', 'old_bottle_effect', 'old_style', 'good_old_days'])

print(f"Reviews mentioning: sulphur={has_sulphur.sum()}, farmyard={has_farmyard.sum()}, overoaked={has_overoaked.sum()}, nostalgia={has_nostalgia.sum()}")

Thresholds: flaw P90=6.8, fruit P75=26.1, peat P75=17.3, structure P75=5.7
Reviews mentioning: sulphur=276, farmyard=320, overoaked=13, nostalgia=465


### D2. Select Candidates by Criterion

Defined in `w2_analysis.ipynb`

def find_reviews(mask, criterion_name, n=3, sort_by=None):
    ...
def term_search(terms):
    return df['review_text'].str.contains(
        r'\b(?:' + '|'.join(re.escape(t) for t in terms) + r')\b',
        case=False, regex=True, na=False
    )

Example: 

- has_overoaked = term_search(['over_oaked', 'planky', 'too_woody', 'over_wooded'])
- has_nostalgia = term_search(['old_school', 'old_bottle_effect', 'old_style', 'good_old_days'])

In [18]:
selections = []

# D1: High score + high flaw
sel = find_reviews(
    (df['score'] >= 90) & (df['flaw_review_text_per1k'] > flaw_p90),
    'D1: High score + high flaw', n=3, sort_by='flaw_review_text_per1k'
)
selections.append(sel)

# D2: Low score + high fruit
sel = find_reviews(
    (df['score'] <= 75) & (df['fruit_review_text_per1k'] > fruit_p75),
    'D2: Low score + high fruit', n=3, sort_by='fruit_review_text_per1k'
)
selections.append(sel)

# D3: High sulphur + high score
sel = find_reviews(
    has_sulphur & (df['score'] >= 85),
    'D3: Sulphur + high score (context-dependent)', n=3, sort_by='score'
)
selections.append(sel)

# D4a: Smoke valued
sel = find_reviews(
    (df['peat_review_text_per1k'] > peat_p75) & (df['score'] >= 88),
    'D4a: Smoke valued', n=2, sort_by='score'
)
selections.append(sel)

# D4b: Smoke condemned
sel = find_reviews(
    (df['peat_review_text_per1k'] > peat_p75) & (df['score'] <= 78),
    'D4b: Smoke condemned', n=2, sort_by='score'
)
selections.append(sel)

# D5a: Farmyard positive
sel = find_reviews(
    has_farmyard & (df['score'] >= 85),
    'D5a: Farmyard as terroir (positive)', n=2, sort_by='score'
)
selections.append(sel)

# D5b: Farmyard negative
sel = find_reviews(
    has_farmyard & (df['score'] <= 78),
    'D5b: Farmyard as unclean (negative)', n=2, sort_by='score'
)
selections.append(sel)

# D6a: Oak as maturity
sel = find_reviews(
    (df['oak_review_text_per1k'] > oak_75) & (df['score'] >= 88),
    'D6a: Oak as maturity', n=2, sort_by='score'
)
selections.append(sel)

# D6b: Oak as excess
sel = find_reviews(
    has_overoaked,
    'D6b: Oak as excess (over-oaked)', n=2
)
selections.append(sel)

# D7: High complexity + low praise
sel = find_reviews(
    (df['structure_review_text_per1k'] > structure_p75) & (df['eval_review_text_per1k'] < eval_median),
    'D7: High complexity + low explicit praise', n=2, sort_by='structure_review_text_per1k'
)
selections.append(sel)

# D8: High praise + low complexity
sel = find_reviews(
    (df['eval_review_text_per1k'] > df['eval_review_text_per1k'].quantile(0.75)) &
    (df['structure_review_text_per1k'] < eval_median),
    'D8: High praise + low complexity', n=2, sort_by='eval_review_text_per1k'
)
selections.append(sel)

# D9: Nostalgia language
sel = find_reviews(
    has_nostalgia,
    'D9: Nostalgia language', n=2
)
selections.append(sel)

# Combine
candidates = pd.concat([s for s in selections if len(s) > 0], ignore_index=True)
candidates = candidates.drop_duplicates(subset='dedupe_hash')  # avoid same review appearing twice
print(); print(f"Total unique candidates selected: {len(candidates)}")

# Print summary table
print(); print("=== Close-Reading Candidates ===")
for i, row in candidates.iterrows():
    print(); print(f"[{row['selection_criterion']}]")
    print(f"  {row['whisky_name_raw']} ({row['distillery']}) — Score: {row['score']}")
    print(f"  Flaw: {row['flaw_review_text_per1k']:.1f}, Fruit: {row['fruit_review_text_per1k']:.1f}, Peat: {row['peat_review_text_per1k']:.1f}")
    # Print first 300 chars of review text
    text = str(row['review_text_original'])[:300]
    print(f"  Text: {text}...")

# Save
candidates.to_csv('data/v2/w2_close_reading_candidates.csv', index=False)
print(); print(f"Saved {len(candidates)} candidates: data/v2/w2_close_reading_candidates.csv")


Total unique candidates selected: 26

=== Close-Reading Candidates ===

[D1: High score + high flaw]
  Port Ellen 22 yo 1976/1999 (55.1%, Signatory Vintage, cask #4795, 282 bottles) (Port Ellen) — Score: 90
  Flaw: 34.5, Fruit: 13.8, Peat: 27.6
  Text: £59.99 a bottle, I remember well. Very expensive, I had thought. Colour: gold. Nose: phoo, there’s this very unusual combination of Demerara sugar with smoked and burnt herbs, plus porridge, soot, new bicycle inner tube, and tarmac. Epitomically Port Ellen this time, with this ‘chemical’ side that’s...

[D1: High score + high flaw]
  Mortlach 31 yo 1987/2018 (54%, Gordon & MacPhail, Connoisseurs Choice, refill sherry hogshead, cask #425, 200 bottles) (Mortlach) — Score: 90
  Flaw: 24.6, Fruit: 4.9, Peat: 4.9
  Text: Look at that lovely colour! Now 1980s vintages could be tricky here and there, so let’s see… Colour: bronze amber. Nose: more dried beef, jerky, pemmican, also a little rubber and struck matchsticks (Mortlach indeed), burnt 

### D3. Print Full Review Texts for Qualitative Reading

In [15]:
print("=" * 80)
print("FULL REVIEW TEXTS FOR CLOSE READING")
print("=" * 80)

for i, row in candidates.iterrows():
    print(); print(f"{'─'*80}")
    print(f"[{row['selection_criterion']}]")
    print(f"Whisky: {row['whisky_name_raw']}")
    print(f"Distillery: {row['distillery']} | Score: {row['score']} | Year: {row['review_year']}")
    print(f"{'─'*80}")
    print(row['review_text_original'])
    print()

FULL REVIEW TEXTS FOR CLOSE READING

────────────────────────────────────────────────────────────────────────────────
[D1: High score + high flaw]
Whisky: Port Ellen 22 yo 1976/1999 (55.1%, Signatory Vintage, cask #4795, 282 bottles)
Distillery: Port Ellen | Score: 90 | Year: 2016
────────────────────────────────────────────────────────────────────────────────
£59.99 a bottle, I remember well. Very expensive, I had thought. Colour: gold. Nose: phoo, there’s this very unusual combination of Demerara sugar with smoked and burnt herbs, plus porridge, soot, new bicycle inner tube, and tarmac. Epitomically Port Ellen this time, with this ‘chemical’ side that’s so unique. With water: fab. Leatherette and supermarket plastic bags, more tyres, more smoky porridge, more pitch and tar. ‘Nosing a brand new Porsche’ (a Ford would do, S.!) Mouth (neat): so unique indeed! Rubber bands marinated in lemonade? Or would that rather be Corsican citron liqueur blended with gorilka vodka? Licking the stree

## Summary: Results → Claims

**Scope:** W2 uses held-out score association as a **validation benchmark**, not as proof of market effects, reader uptake, or causal value creation. Findings characterize **this Whiskyfun corpus and reviewing practice** only.

### Claim 1: Expert valuation is not generic sentiment (E1)
- **Evidence:** Task C R² comparison — M0 (length + year) Adj-R² ≈ 0.103; M1 (VADER) ≈ 0.161; M2 (full dictionary, 11 cats) ≈ 0.290; M3 (dictionary minus `eval`, 10 cats) ≈ 0.265
- Domain vocabulary recovers substantially more held-out score variation than generic sentiment and baseline controls
- M3 retains most of M2 (ΔAdj-R² ≈ +0.024 from dropping explicit eval) → sensory and structural terms already carry evaluative signal, not only verdict language
- **Interpretation:** Supports a language of **discernment** beyond positive/negative valence; prediction benchmarks validity, it is not the sociological explanation

### Claim 1b: The dictionary tracks independently specified style cues (construct validity)
- **Evidence:** Task B known-groups validation (feature-defined masks, not score-defined)
- **Peated** (peat > P75): peat language d ≈ 2.58; less fruit/floral; more structure
- **Sherried** (sherry > P75): sherry d ≈ 2.66; less peat/fruit in discourse
- **Bourbon-cask proxy** (oak > P75 and sherry < median): oak d ≈ 1.69; sherry d ≈ −1.06
- **Interpretation:** Categories behave as measurable style distinctions in expert text, not only as score correlates

### Claim 2: High value is organized through structure and low defect language, not uniform “pleasantness”
- **Evidence:** Task A1 coefficients — `structure` and `texture` positive; **`flaw` strongest negative predictor** (β ≈ −0.33 per 1k tokens)
- **Criterion associations** (Task B4–B5; score-defined groups — validation of score–text link, not independent construct proof):
  - High-score (≥90): lower flaw (0.87 vs 2.07), higher structure, texture, and eval; **more peat** (d ≈ 0.41) — high value is not simply “less intensity”
  - Low-score (≤75): elevated flaw (8.62 vs 1.52, d ≈ 0.86); lower structure and eval
- **Interpretation:** Valuation in this corpus is partly a bundle of complexity/texture and **absence of defect classification**, not generic pleasure alone

### Claim 3: Defects are culturally organized (boundary proposition / E3, qualitative)
- **Evidence:** Quantitative — flaw ↓ with score in regression and high/low-score contrasts; qualitative — Task D close-reading candidates
- Same lexemes cross the valued/devalued boundary by context: sulphur, farmyard, smoke, oak (e.g. D1 “desirable flaws”; D5a farmyard as terroir vs D5b as unclean; D6a oak-as-maturity vs D6b over-oaked)
- **Interpretation:** Material sensations become evaluatively consequential through **context-dependent defect vs character** judgments (Douglas/Lamont), not fixed one-to-one term meanings

### Claim 4: Evaluation is embedded in structured sensory description (E2 / judgment device)
- **Evidence:** Task A3 section-level R² — NMF (Nose+ Mouth+Finish) Adj-R² ≈ 0.336; Full Text ≈ 0.289; Nose ≈ 0.208; Mouth ≈ 0.234; Finish ≈ 0.130; **Comments ≈ 0.022**
- Sensory sections carry most score signal; the Comments block adds little on the complete-case sample
- **Interpretation:** The review form (Nose/Mouth/Finish) is where classification and comparison happen before the numerical score — evaluation is **textually legitimated throughout** the judgment device, not confined to explicit verdict prose

### Limits (do not overclaim)
- Single expert institution; no reader class position, market causation, or proof of Bourdieu’s homology
- High/low-score groups are **criterion associations**, not independent validation of the instrument
- Embedding-based Natural/Artificial boundary tests (w4) extend E3 beyond this notebook
- Interpretable dictionary is substantively useful but does not exhaust evaluative discourse (broad lexical benchmarks expected to predict more)